In [ ]:
!pip install -q resampy

import os
import time
import zipfile
import warnings

import gdown
import librosa
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
%matplotlib inline

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Версия PyTorch:", torch.__version__)
print("Устройство:", device)


In [ ]:
TESS_URL = "https://storage.yandexcloud.net/aiueducation/Content/base/l12/dataverse_files.zip"
SAVEE_URL = "https://storage.yandexcloud.net/aiueducation/Content/base/l12/archive.zip"

TESS_ZIP = "tess_data.zip"
SAVEE_ZIP = "savee_data.zip"
TESS_DIR = "./tess_dataset"
SAVEE_DIR = "./savee_dataset"


def load_archive(url, archive_name):
    if not os.path.exists(archive_name):
        gdown.download(url, archive_name, quiet=True)


def extract_archive(archive_name, folder_name):
    os.makedirs(folder_name, exist_ok=True)
    with zipfile.ZipFile(archive_name, "r") as file:
        file.extractall(folder_name)
    print(f"Архив {archive_name} распакован в {folder_name}")


load_archive(TESS_URL, TESS_ZIP)
load_archive(SAVEE_URL, SAVEE_ZIP)

extract_archive(TESS_ZIP, TESS_DIR)
extract_archive(SAVEE_ZIP, SAVEE_DIR)


In [ ]:
EMOTIONS = {
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "happy": 3,
    "neutral": 4,
    "sad": 5,
    "surprise": 6,
}

NUM_CLASSES = len(EMOTIONS)
CLASS_NAMES = list(EMOTIONS.keys())
INDEX_TO_EMOTION = {value: key for key, value in EMOTIONS.items()}


def make_audio_features(path):
    signal, sample_rate = librosa.load(path, res_type="kaiser_fast")

    mfcc = librosa.feature.mfcc(y=signal, sr=sample_rate, n_mfcc=40)
    mfcc_avg = mfcc.mean(axis=1)
    mfcc_dev = mfcc.std(axis=1)

    mel = librosa.feature.melspectrogram(y=signal, sr=sample_rate)
    mel_avg = mel.mean(axis=1)
    mel_dev = mel.std(axis=1)

    chroma = librosa.feature.chroma_stft(y=signal, sr=sample_rate)
    chroma_avg = chroma.mean(axis=1)
    chroma_dev = chroma.std(axis=1)

    return np.concatenate([mfcc_avg, mfcc_dev, mel_avg, mel_dev, chroma_avg, chroma_dev])


In [ ]:
def read_tess_label(name):
    lowered = name.lower()
    variants = [
        ("angry", "angry"),
        ("disgust", "disgust"),
        ("fear", "fear"),
        ("happy", "happy"),
        ("neutral", "neutral"),
        ("sad", "sad"),
    ]
    for marker, label in variants:
        if marker in lowered:
            return label
    if "ps" in lowered or "surprise" in lowered:
        return "surprise"
    return None


def build_tess_table(root_dir):
    data = []
    target = []

    for folder, _, files in os.walk(root_dir):
        for filename in files:
            if not filename.lower().endswith(".wav"):
                continue
            emotion = read_tess_label(filename)
            if emotion is None:
                continue
            full_path = os.path.join(folder, filename)
            data.append(make_audio_features(full_path))
            target.append(EMOTIONS[emotion])

    return np.asarray(data, dtype=np.float32), np.asarray(target, dtype=np.int64)


print("Разбор датасета TESS...")
start_time = time.time()

x_tess, y_tess = build_tess_table(TESS_DIR)

print(f"Файлов обработано: {len(x_tess)}")
print(f"Время обработки: {round(time.time() - start_time)} сек.")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x_tess,
    y_tess,
    test_size=0.2,
    stratify=y_tess,
    random_state=SEED,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

y_train = y_train.astype(np.int64)
y_test = y_test.astype(np.int64)

train_data = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)

test_data = TensorDataset(
    torch.tensor(X_test_scaled, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long),
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, drop_last=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print("Обучающая выборка:", X_train_scaled.shape)
print("Тестовая выборка:", X_test_scaled.shape)


In [ ]:
class EmotionDenseNet(nn.Module):
    def __init__(self, input_size, class_count):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, class_count),
        )

    def forward(self, x):
        return self.layers(x)


model = EmotionDenseNet(X_train_scaled.shape[1], NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)
print("Количество параметров:", sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.set_grad_enabled(is_train):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * batch_y.size(0)
            total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
            total_count += batch_y.size(0)

    return total_loss / total_count, total_correct / total_count


history = {
    "loss": [],
    "accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

EPOCHS = 30

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_accuracy = run_epoch(model, test_loader, criterion)

    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Эпоха {epoch:02d}/{EPOCHS} | "
        f"loss={train_loss:.4f} | accuracy={train_accuracy:.4f} | "
        f"val_loss={val_loss:.4f} | val_accuracy={val_accuracy:.4f}"
    )


In [ ]:
def plot_training_result(train_history):
    fig, axes = plt.subplots(1, 2, figsize=(20, 5))
    fig.suptitle("Обучение классификатора эмоций")

    axes[0].plot(train_history["accuracy"], label="Обучение")
    axes[0].plot(train_history["val_accuracy"], label="Тест")
    axes[0].set_xlabel("Эпоха")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(train_history["loss"], label="Обучение")
    axes[1].plot(train_history["val_loss"], label="Тест")
    axes[1].set_xlabel("Эпоха")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True)

    plt.show()


plot_training_result(history)


In [ ]:
def predict_labels(model, data_array, batch_size=64):
    model.eval()
    predictions = []
    dataset = TensorDataset(torch.tensor(data_array, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (batch_x,) in loader:
            logits = model(batch_x.to(device))
            predictions.extend(logits.argmax(dim=1).cpu().numpy())

    return np.asarray(predictions, dtype=np.int64)


def show_confusion(y_true, y_pred, title, color_map):
    matrix = confusion_matrix(y_true, y_pred)
    display = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=CLASS_NAMES,
    )

    fig, ax = plt.subplots(figsize=(8, 8))
    display.plot(ax=ax, cmap=color_map, xticks_rotation=45)
    plt.title(title)
    plt.show()


test_loss, test_accuracy = run_epoch(model, test_loader, criterion)

print("=" * 60)
print(f"ИТОГОВАЯ ТОЧНОСТЬ НА ТЕСТОВОМ НАБОРЕ TESS: {round(test_accuracy * 100, 2)}%")
print("Выполнение условия технического задания (>= 98%):", "ВЫПОЛНЕНО!" if test_accuracy >= 0.98 else "НЕ ВЫПОЛНЕНО.")
print("=" * 60)

tess_predicted = predict_labels(model, X_test_scaled)

show_confusion(
    y_true=y_test,
    y_pred=tess_predicted,
    title="Матрица ошибок для датасета TESS",
    color_map="Blues",
)


In [ ]:
SAVEE_CODES = {
    "a": "angry",
    "d": "disgust",
    "f": "fear",
    "h": "happy",
    "n": "neutral",
    "sa": "sad",
    "su": "surprise",
}


def read_savee_label(filename):
    base = os.path.splitext(filename)[0]
    parts = base.split("_")
    if len(parts) < 2:
        return None
    code = "".join(item for item in parts[1] if item.isalpha())
    return SAVEE_CODES.get(code)


def build_savee_table(root_dir):
    data = []
    target = []

    for folder, _, files in os.walk(root_dir):
        for filename in files:
            if not filename.lower().endswith(".wav"):
                continue
            emotion = read_savee_label(filename)
            if emotion is None:
                continue
            full_path = os.path.join(folder, filename)
            data.append(make_audio_features(full_path))
            target.append(EMOTIONS[emotion])

    return np.asarray(data, dtype=np.float32), np.asarray(target, dtype=np.int64)


print("Разбор датасета SAVEE...")
x_savee, y_savee = build_savee_table(SAVEE_DIR)

print(f"Файлов обработано: {len(x_savee)}")


In [ ]:
x_savee_scaled = scaler.transform(x_savee).astype(np.float32)
y_savee = y_savee.astype(np.int64)

savee_data = TensorDataset(
    torch.tensor(x_savee_scaled, dtype=torch.float32),
    torch.tensor(y_savee, dtype=torch.long),
)

savee_loader = DataLoader(savee_data, batch_size=64, shuffle=False)
savee_loss, savee_accuracy = run_epoch(model, savee_loader, criterion)

print("\n" + "=" * 60)
print(f"ТОЧНОСТЬ КЛАССИФИКАТОРА НА ДАННЫХ SAVEE: {round(savee_accuracy * 100, 2)}%")
print("=" * 60)

savee_predicted = predict_labels(model, x_savee_scaled)

show_confusion(
    y_true=y_savee,
    y_pred=savee_predicted,
    title="Матрица ошибок на стороннем датасете SAVEE",
    color_map="Reds",
)
